<h3>실습 안내</h3><p>설정 정의, 값 하나 변경, 색인 해시, 캐시 판단, 검색, JSONL 기록을 한 행동씩 실행합니다.</p>

In [1]:
# 실제 임베딩·벡터 검색·청크 재생성에 필요한 패키지를 설치합니다.
# ==는 버전을 고정하고 -q는 설치 로그를 줄입니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


<h3>1. API 키 준비</h3><p><code>.env</code>를 로드해 실제 임베딩 색인을 만들 준비를 합니다.</p>

In [2]:
# find_dotenv()는 .env 경로를 찾고 load_dotenv()는 그 파일의 값을 로드합니다.
# API 키가 없으면 assert가 실제 외부 호출 전에 실행을 멈춥니다.
import os
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, ".env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
print("API 키 준비 완료")

API 키 준비 완료


<h3>2. 기본 설정을 딕셔너리로 보기</h3><p>분할·임베딩·검색 설정을 중첩 딕셔너리 하나에 모으고 각 값을 직접 출력합니다.</p>

In [3]:
# config의 바깥 키는 단계, 안쪽 키는 그 단계의 세부 설정입니다.
# 청크 크기·임베딩 모델·검색 개수를 따로 읽어 중첩 구조를 확인합니다.
config = {
    "split": {"chunk_size": 500, "chunk_overlap": 50},
    "embedding": {"model": "text-embedding-3-small"},
    "retrieval": {"k": 1},
}
print("chunk_size:", config["split"]["chunk_size"])
print("embedding model:", config["embedding"]["model"])
print("k:", config["retrieval"]["k"])

chunk_size: 500
embedding model: text-embedding-3-small
k: 1


<h3>3. 검색 k만 변경</h3><p><code>deepcopy()</code>로 원본을 보존한 새 설정을 만들고 <code>retrieval.k</code>만 2로 바꾹니다.</p>

In [4]:
# deepcopy()는 안쪽 딕셔너리까지 복제해 원본 config의 변경을 막습니다.
# changed_k에서 retrieval.k만 바꾸고 원본과 변경 값을 나란히 보여 줍니다.
from copy import deepcopy

changed_k = deepcopy(config)
changed_k["retrieval"]["k"] = 2
print("기본 k:", config["retrieval"]["k"])
print("변경 k:", changed_k["retrieval"]["k"])
assert config["split"] == changed_k["split"]
assert config["embedding"] == changed_k["embedding"]

기본 k: 1
변경 k: 2


<h3>4. 색인 설정만 JSON으로 직렬화</h3><p>청크와 임베딩 설정은 색인을 바꾸지만 검색 <code>k</code>는 기존 색인을 재사용할 수 있으므로 해시 입력에서 제외합니다.</p>

In [5]:
# index_settings에는 색인 결과에 영향을 주는 split과 embedding만 담습니다.
# sort_keys=True는 키 순서를 고정해 같은 설정이 같은 JSON 문자열이 되게 합니다.
import json

index_settings = {
    "split": config["split"],
    "embedding": config["embedding"],
}
index_json = json.dumps(index_settings, sort_keys=True)
print(index_json)

{"embedding": {"model": "text-embedding-3-small"}, "split": {"chunk_overlap": 50, "chunk_size": 500}}


<h3>5. JSON 문자열을 설정 해시로 변환</h3><p>SHA-256은 긴 바이트 입력을 고정 길이 문자열로 바꾹니다. 실습에서는 앞의 10자를 캐시 키로 사용합니다.</p>

In [6]:
# encode()는 JSON 문자열을 해시 함수가 받는 바이트로 바꿉니다.
# hexdigest()는 SHA-256 결과를 16진수 문자열로 반환하며 [:10]은 앞 10자만 선택합니다.
import hashlib

index_bytes = index_json.encode()
full_hash = hashlib.sha256(index_bytes).hexdigest()
index_key = full_hash[:10]
print("전체 해시:", full_hash)
print("캐시 키:", index_key)

전체 해시: d853036bf1a4f0d7a105162822756cc9abd0fe816392ecfb97046374ab1fa126
캐시 키: d853036bf1


<h3>6. 실제 임베딩 색인 생성</h3><p>짧은 Document 두 건을 임베딩해 벡터스토어를 만들고 해시를 키로 캐시에 보관합니다.</p>

In [7]:
# documents는 검색 실험에 사용할 두 Document입니다.
# cache는 설정 해시를 키로, 실제 벡터스토어 객체를 값으로 보관합니다.
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(page_content="휴관일은 매주 월요일입니다.", metadata={"doc_id": "D01"}),
    Document(page_content="대출 기간은 14일입니다.", metadata={"doc_id": "D02"}),
]
embedding = OpenAIEmbeddings(model=config["embedding"]["model"])
base_splitter = RecursiveCharacterTextSplitter(**config["split"])
indexed_documents = base_splitter.split_documents(documents)
store = InMemoryVectorStore.from_documents(indexed_documents, embedding)
cache = {index_key: store}
print("기본 청크 수:", len(indexed_documents))
print("캐시 키:", list(cache.keys()))

기본 청크 수: 2
캐시 키: ['d853036bf1']


<h3>7. k 변경 설정의 캐시 재사용 판단</h3><p><code>changed_k</code>의 분할과 임베딩 설정이 같으므로 해시도 같은지 확인합니다.</p>

In [8]:
# changed_index_settings는 changed_k에서 색인에 영향을 주는 두 부분만 고릅니다.
# changed_key가 cache에 있으면 비용이 드는 문서 재임베딩을 생략할 수 있습니다.
changed_index_settings = {
    "split": changed_k["split"],
    "embedding": changed_k["embedding"],
}
changed_json = json.dumps(changed_index_settings, sort_keys=True)
changed_hash = hashlib.sha256(changed_json.encode()).hexdigest()
changed_key = changed_hash[:10]
cache_hit = changed_key in cache
print("기본 키:", index_key)
print("변경 k 키:", changed_key)
print("캐시 재사용:", cache_hit)
assert cache_hit

기본 키: d853036bf1
변경 k 키: d853036bf1
캐시 재사용: True


<h3>8. 재사용한 store로 검색</h3><p>캐시에서 벡터스토어를 가져온 뒤 변경한 <code>k=2</code>를 검색 인자로 사용합니다.</p>

In [9]:
# reused_store는 cache에 보관했던 기존 InMemoryVectorStore 객체입니다.
# is는 값 비교가 아니라 두 변수가 같은 객체를 가리키는지 검사합니다.
reused_store = cache[changed_key]
search_k = changed_k["retrieval"]["k"]
hits = reused_store.similarity_search("휴관일은 언제인가요?", k=search_k)
retrieved_ids = []
for document in hits:
    retrieved_ids.append(document.metadata["doc_id"])
print("같은 store:", reused_store is store)
print("검색 k:", search_k)
print("검색 ID:", retrieved_ids)

같은 store: True
검색 k: 2
검색 ID: ['D01', 'D02']


<h3>9. 기록할 JSON 한 행 만들기</h3><p>설정 이름, 해시, 캐시 적중 여부, 검색 ID를 딕셔너리 하나에 모윽니다.</p>

In [10]:
# record는 한 번의 실험 설정과 결과를 함께 담은 딕셔너리입니다.
# ensure_ascii=False는 한글을 유니코드 이스케이프 대신 읽을 수 있는 문자로 표시합니다.
record = {
    "name": "k2",
    "config_sha": changed_key,
    "cache_hit": cache_hit,
    "retrieved": retrieved_ids,
}
record_json = json.dumps(record, ensure_ascii=False)
print(record_json)

{"name": "k2", "config_sha": "d853036bf1", "cache_hit": true, "retrieved": ["D01", "D02"]}


<h3>10. JSONL에 한 행 추가하고 다시 읽기</h3><p>임시 폴더를 사용하므로 수업 폴더에 실험 파일을 남기지 않습니다.</p>

In [11]:
# 'a' 모드는 기존 파일 끝에 새 행을 추가합니다.
# splitlines()는 JSONL을 행별로 나누고 json.loads()는 한 행을 딕셔너리로 복원합니다.
from pathlib import Path
from tempfile import TemporaryDirectory

temp_dir = TemporaryDirectory()
log_path = Path(temp_dir.name) / "runs.jsonl"
with log_path.open("a", encoding="utf-8") as file:
    file.write(record_json + "\n")
saved_records = []
for line in log_path.read_text(encoding="utf-8").splitlines():
    saved_records.append(json.loads(line))
print(saved_records)
assert saved_records[0] == record

[{'name': 'k2', 'config_sha': 'd853036bf1', 'cache_hit': True, 'retrieved': ['D01', 'D02']}]


<h3>정리</h3><p>설정 하나를 바꾸고, 색인 해시로 캐시 재사용을 판단하고, 검색 결과를 JSONL 한 행으로 저장했습니다.</p>

<p><strong>원인</strong> — 청크 설정을 바꿔 색인 해시가 달라졌는데도 기존 <code>index_key</code>로 store를 가져왔습니다.</p>

In [12]:
# changed_split은 청크 크기와 겹침을 바꾼 새 설정입니다.
# wrong_store는 새 해시가 아닌 기존 해시로 가져왔으므로 잘못된 재사용입니다.
changed_split = deepcopy(config)
changed_split["split"]["chunk_size"] = 20
changed_split["split"]["chunk_overlap"] = 2
wrong_store = cache[index_key]
print("잘못된 재사용:", wrong_store is store)
new_settings = {"split": changed_split["split"], "embedding": changed_split["embedding"]}
new_json = json.dumps(new_settings, sort_keys=True)
new_key = hashlib.sha256(new_json.encode()).hexdigest()[:10]
try:
    assert new_key in cache
except AssertionError:
    print("예상한 실패: 변경된 설정의 색인이 없습니다.")

잘못된 재사용: True
예상한 실패: 변경된 설정의 색인이 없습니다.


<p><strong>수정</strong> — 변경된 청크 설정으로 문서를 다시 나누고, 실제 임베딩 색인을 새 해시에 저장합니다.</p>

In [13]:
# **changed_split['split']은 딕셔너리를 chunk_size와 chunk_overlap 키워드 인자로 풀어 줍니다.
# rebuilt_store는 새 청크를 실제로 재임베딩한 다른 객체입니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

new_splitter = RecursiveCharacterTextSplitter(**changed_split["split"])
rebuilt_documents = new_splitter.split_documents(documents)
rebuilt_store = InMemoryVectorStore.from_documents(rebuilt_documents, embedding)
cache[new_key] = rebuilt_store
rebuilt_hits = rebuilt_store.similarity_search("휴관일", k=1)
print("After 청크 수:", len(rebuilt_documents))
print("After 검색:", rebuilt_hits[0].metadata["doc_id"])
assert rebuilt_store is not store
assert new_key in cache

After 청크 수: 2
After 검색: D01
